In [4]:
import sys
import os

# 1. Define the path to your folder
# 🛑 CHANGE 'src' IF YOUR FOLDER HAS A DIFFERENT NAME
folder_name = 'src' 
folder_path = os.path.join(os.getcwd(), folder_name)

# 2. Add this folder to Python's "Search Path"
if folder_path not in sys.path:
    sys.path.append(folder_path)
    print(f"✅ Added '{folder_path}' to system path.")

# 3. NOW you can import directly from helper
try:
    from helper import get_pinecone_vectorstore, load_medical_model, build_rag_prompt, generate_answer
    print("✅ Successfully imported helper functions!")
except ImportError as e:
    print(f"❌ Error: {e}")
    print(f"Double check that 'helper.py' is actually inside the '{folder_name}' folder.")

✅ Added 'c:\Users\khali\OneDrive\Bureau\medical rag\src' to system path.
✅ Successfully imported helper functions!


In [10]:
import os
import torch
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
from dotenv import load_dotenv


load_dotenv()

# --- 🛑 UPDATE THESE PATHS ---
# Make sure these folders actually exist on your PC!
ADAPTER_PATH = r"C:\Users\khali\OneDrive\Bureau\mistral fine tuned\mistral-medical-adapter-final"
BASE_PATH = r"c:\Users\khali\OneDrive\Bureau\mistral fine tuned\Mistral-7B-Instruct-v0.2"
INDEX_NAME = "medical-chatbot"

# Global Storage
models = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("🚀 Starting Local Medical RAG...")
    
    api_key = os.getenv("PINECONE_API_KEY") # Ensure .env has PINECONE_API_KEY=...
    
    try:
        # 1. Load Pinecone
        print("🔌 Connecting to Pinecone...")
        models["docsearch"] = get_pinecone_vectorstore(api_key, INDEX_NAME)
        
        # 2. Load Local Models
        print("🧠 Loading Mistral from disk...")
        tokenizer, model = load_medical_model(BASE_PATH, ADAPTER_PATH)
        models["tokenizer"] = tokenizer
        models["model"] = model
        print("✅ System Ready!")
        
    except Exception as e:
        print(f"❌ Startup Error: {e}")
    
    yield
    
    # Cleanup
    print("🛑 Shutting down...")
    models.clear()
    torch.cuda.empty_cache()

app = FastAPI(lifespan=lifespan)

# Mounts
if os.path.exists("static"):
    app.mount("/static", StaticFiles(directory="static"), name="static")
templates = Jinja2Templates(directory="templates")

@app.get("/", response_class=HTMLResponse)
async def read_root(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@app.post("/chat")
async def chat_endpoint(query: str = Form(...)):
    print(f"🔎 User: {query}")
    try:
        # 1. Retrieve
        results = models["docsearch"].similarity_search(query, k=2)
        context = "\n\n".join([d.page_content for d in results])
        
        # 2. Prompt
        prompt = build_rag_prompt(query, context)
        
        # 3. Generate
        answer = generate_answer(models["model"], models["tokenizer"], prompt)
        
        return JSONResponse({"response": answer})
    except Exception as e:
        return JSONResponse({"response": f"Error: {str(e)}"})

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 14
python-dotenv could not parse statement starting at line 15
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 18
python-dotenv could not parse statement starting at line 19


In [11]:
# ------------------------------------------------------------------
# CELL 2: START SERVER (JUPYTER COMPATIBLE)
# ------------------------------------------------------------------
import uvicorn
import asyncio

# Apply nest_asyncio just in case
import nest_asyncio
nest_asyncio.apply()

print("🌐 Starting Server at http://127.0.0.1:8000")
print("ℹ️  Stop the cell (Square Button) to kill the server.")

# Configure the server
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)

# Run the server in the existing notebook loop
# We use 'await' here instead of uvicorn.run()
await server.serve()

🌐 Starting Server at http://127.0.0.1:8000
ℹ️  Stop the cell (Square Button) to kill the server.


INFO:     Started server process [70272]
INFO:     Waiting for application startup.


🚀 Starting Local Medical RAG...
🔌 Connecting to Pinecone...
🧠 Loading Mistral from disk...
⏳ Loading Tokenizer from c:\Users\khali\OneDrive\Bureau\mistral fine tuned\Mistral-7B-Instruct-v0.2...


`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


⏳ Loading Base Model...


...

⏳ Loading LoRA Adapter from C:\Users\khali\OneDrive\Bureau\mistral fine tuned\mistral-medical-adapter-final...


INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


✅ System Ready!
INFO:     127.0.0.1:39490 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:39490 - "GET /static/style.css HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:39490 - "POST /search HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:51297 - "GET / HTTP/1.1" 200 OK
🔎 User: i have headache
INFO:     127.0.0.1:22010 - "POST /chat HTTP/1.1" 200 OK
🔎 User: i have acne
INFO:     127.0.0.1:29497 - "POST /chat HTTP/1.1" 200 OK
🔎 User: now i have a mild headache in my right half of head what should i do
INFO:     127.0.0.1:43741 - "POST /chat HTTP/1.1" 200 OK
🔎 User: what is Actinomyces israelii
INFO:     127.0.0.1:51778 - "POST /chat HTTP/1.1" 200 OK
🔎 User: what is Actinomyces israelii and is it dangerous , is it treatable?
INFO:     127.0.0.1:45725 - "POST /chat HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [70272]


🛑 Shutting down...
